# Lab 2.3 — Implementing A2C (Advantage Actor–Critic) from Scratch

**Intelligent Agents · UC2 · Policy Gradient Methods**

**Student name:** ______________________ **Date:** ______________

## Learning objectives

By the end of this lab you will be able to:

1. **Measure the variance problem** of Monte-Carlo returns (lecture §01).
2. Build an **actor–critic** network: a policy $\pi_\theta(a|s)$ and a value function $V_\psi(s)$ (lecture §02).
3. Compute the **TD error** $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ and use it as an advantage estimate (lecture §03).
4. Compute **n-step returns and advantages** with a backward pass (lecture §03).
5. Implement the **A2C loss** — policy + value + entropy — and the **one-step update** (lecture §04).
6. Train A2C on **CartPole-v1** with parallel environments, and study the effect of $n$ and of the learning rate.

### The algorithm you will implement (slide 18)

```
Initialize actor θ, critic ψ; observe s in all N envs
repeat
  for t = 0 … n−1 (all N envs at once):            ← Part 6  (Collect)
      aₜ ~ πθ(·|sₜ);  step → rₜ, sₜ₊₁, doneₜ
  G ← Vψ(sₙ)  (0 if the episode ended)              ← Part 4  (Bootstrap)
  for t = n−1 … 0:
      G ← rₜ + γ·G·(1 − doneₜ);   Gₜ ← G             ← Part 4  (n-step returns)
      Âₜ ← Gₜ − Vψ(sₜ)                               ← Parts 3–4
  L ← −Σ log πθ(aₜ|sₜ)·Âₜ + c_v Σ (Gₜ − Vψ(sₜ))² − c_e Σ H(πθ(·|sₜ))   ← Part 5
  one gradient step on L (θ and ψ); discard the batch                   ← Part 7
```

### How this lab is organised

| Part | Topic (slides) | TODOs | Points |
|---|---|---|---|
| 1 | The variance problem (slides 4–5) | 1.1 | 10 |
| 2 | Actor–critic network (slide 8) | 2.1, 2.2 | 15 |
| 3 | TD error as advantage (slide 13) | 3.1 | 10 |
| 4 | n-step returns and advantages (slide 14) | 4.1 | 20 |
| 5 | The A2C loss (slide 17) | 5.1 | 15 |
| 6 | Collecting a rollout (slides 16, 19) | 6.1 | 10 |
| 7 | The A2C update and training (slides 18–19) | 7.1 | 10 |
| 8 | Experiments: n and the learning rate (slides 14, 21) | 8.1, 8.2 | 10 |
| | **Total** | | **100** |

Every TODO is followed by a **✅ check cell** that prints `PASSED` when your code is correct.

## 0 · Setup
On Google Colab, PyTorch is pre-installed; the first cell installs Gymnasium if needed.

In [ ]:
import importlib, subprocess, sys
for pkg in ["gymnasium", "torch", "matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
import random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt

def set_seed(seed: int = 1):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(1)
print("torch", torch.__version__, "| gymnasium", gym.__version__)

### 0.1 · The environment: CartPole-v1
State: cart position, cart velocity, pole angle, pole angular velocity. Actions: push left / push right. Reward: +1 per step the pole stays up; maximum return **500**.

### 0.2 · Provided helper: parallel environments
`MultiEnv` steps `n` copies of the environment together (the **synchronous workers** of slide 16) and auto-resets finished episodes, so the observation returned after a `done` is the first observation of the next episode. It records the return of every finished episode. *You do not need to modify it.*

In [ ]:
class MultiEnv:
    # Steps n copies of an environment in lockstep and auto-resets finished episodes.
    def __init__(self, env_id: str, n: int, seed: int = 0):
        self.envs = [gym.make(env_id) for _ in range(n)]
        self.n, self.seed = n, seed
        self.obs_dim = self.envs[0].observation_space.shape[0]
        self.n_actions = self.envs[0].action_space.n
        self.ep_return = np.zeros(n)
        self.finished_returns = []

    def reset(self):
        obs = [e.reset(seed=self.seed + i)[0] for i, e in enumerate(self.envs)]
        return np.array(obs, dtype=np.float32)

    def step(self, actions):
        obs, rewards, dones = [], [], []
        for i, (e, a) in enumerate(zip(self.envs, actions)):
            o, r, terminated, truncated, _ = e.step(int(a))
            self.ep_return[i] += r
            done = terminated or truncated
            if done:
                self.finished_returns.append(self.ep_return[i])
                self.ep_return[i] = 0.0
                o, _ = e.reset()
            obs.append(o); rewards.append(r); dones.append(float(done))
        return (np.array(obs, dtype=np.float32),
                np.array(rewards, dtype=np.float32),
                np.array(dones, dtype=np.float32))

    def close(self):
        for e in self.envs: e.close()

---
## Part 1 · The variance problem  *(10 pts)*

> **Slides 4–5.** REINFORCE scores every action with the **Monte-Carlo return** $G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots$ of the rest of the episode. Because the environment and the policy are random, returns from the *same* starting state vary a lot.

### TODO 1.1 — `discounted_returns`
Given the list of rewards of **one complete episode**, return a NumPy array with $G_t$ for every step. Compute it **backward**: start with `G = 0` and, for `t` from the last step down to 0, set `G = rewards[t] + gamma * G`.

In [ ]:
def discounted_returns(rewards, gamma=0.99):
    # ===== TODO 1.1 =====
    raise NotImplementedError("TODO 1.1: compute the discounted returns backward")
    # ===== END TODO =====

In [ ]:
# ✅ Check 1.1
g = discounted_returns([1.0, 1.0, 1.0], gamma=0.9)
print(g, " expected [2.71 1.9  1.  ]")
assert np.allclose(g, [2.71, 1.9, 1.0])
print("PASSED ✔ Check 1.1")

### 📈 See the variance
The cell below plays 200 episodes of a **fixed random policy** (50 % left / 50 % right), always starting from the *same* initial state, and plots the distribution of $G_0$. This is the noise REINFORCE has to learn from.

In [ ]:
env = gym.make("CartPole-v1")
G0 = []
rng = np.random.default_rng(0)
for ep in range(200):
    obs, _ = env.reset(seed=42)            # same initial state every time
    rewards, done = [], False
    while not done:
        obs, r, term, trunc, _ = env.step(int(rng.integers(2)))
        rewards.append(r); done = term or trunc
    G0.append(discounted_returns(rewards, gamma=0.99)[0])
env.close()
G0 = np.array(G0)
print(f"G0 over 200 episodes: mean = {G0.mean():.2f}, std = {G0.std():.2f}, min = {G0.min():.2f}, max = {G0.max():.2f}")
plt.figure(figsize=(7, 3.2)); plt.hist(G0, bins=25, color="#7FA6E8", edgecolor="#14213D")
plt.title("Return from the same start state, same policy"); plt.xlabel("G₀"); plt.ylabel("episodes"); plt.show()

**Question 1.** Every episode starts from the same state with the same policy. Why are the returns so different, and why is this a problem for REINFORCE?

*Your answer:* …

---
## Part 2 · The actor–critic network  *(15 pts)*

> **Slide 8.** The **actor** $\pi_\theta(a|s)$ (the player) outputs a probability for each action. The **critic** $V_\psi(s)$ (the coach) outputs one number.

```
state s ──► Linear(obs,64) ─ tanh ─ Linear(64,64) ─ tanh ─ Linear(64, n_actions) ──► logits ──► Categorical ──► a
        └─► Linear(obs,64) ─ tanh ─ Linear(64,64) ─ tanh ─ Linear(64, 1)         ──► V(s)
```
`layer_init` (provided) uses orthogonal initialisation; `std=0.01` on the actor's output makes the initial policy almost uniform.

### TODO 2.1 — build `self.critic` and `self.actor`
### TODO 2.2 — `get_action_and_value(x, action=None)`
Build `Categorical(logits=...)`, sample an action if none is given, and return `action, log_prob(action), entropy(), value` (all of shape `(batch,)`).

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias_const)
    return layer

In [ ]:
class Agent(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        # ===== TODO 2.1 =====
        raise NotImplementedError("TODO 2.1: build self.critic and self.actor")
        # ===== END TODO =====

    def get_value(self, x):
        return self.critic(x).squeeze(-1)

    def get_action_and_value(self, x, action=None):
        # ===== TODO 2.2 =====
        raise NotImplementedError("TODO 2.2: implement get_action_and_value")
        # ===== END TODO =====

In [ ]:
# ✅ Check Part 2
set_seed(0)
_agent = Agent(obs_dim=4, n_actions=2)
_x = torch.randn(8, 4)
_a, _lp, _ent, _v = _agent.get_action_and_value(_x)
assert _a.shape == _lp.shape == _ent.shape == _v.shape == (8,), "all outputs must have shape (batch,)"
_p = sum(torch.exp(_agent.get_action_and_value(_x, torch.full((8,), k, dtype=torch.long))[1]) for k in range(2))
assert torch.allclose(_p, torch.ones(8), atol=1e-5), "action probabilities must sum to 1"
assert torch.allclose(_ent, torch.full((8,), np.log(2)), atol=1e-2), "initial policy should be ~uniform (did you use std=0.01?)"
print("PASSED ✔ Part 2")

---
## Part 3 · The TD error as an advantage  *(10 pts)*

> **Slide 13.** $A(s,a) = Q(s,a) - V(s)$ and, by Bellman, $Q(s,a) \approx r + \gamma V(s')$. So a single critic is enough:
> $$\hat A_t \approx \delta_t = r_t + \gamma V(s_{t+1})\,(1 - d_t) - V(s_t)$$
> where $d_t = 1$ if the episode ended at step $t$ (then there is no next state to bootstrap from).

### TODO 3.1 — `td_error`
Works element-wise on tensors (or floats).

In [ ]:
def td_error(reward, value, next_value, done, gamma=0.99):
    # ===== TODO 3.1 =====
    raise NotImplementedError("TODO 3.1: return the TD error")
    # ===== END TODO =====

In [ ]:
# ✅ Check Part 3 — the two cases from slide 13 and the recap quiz
d1 = td_error(2.0, 10.0, 12.0, 0.0, gamma=0.9)
d2 = td_error(0.0, 10.0, 9.0, 0.0, gamma=0.9)
d3 = td_error(1.0, 5.0, 4.0, 0.0, gamma=0.9)
d4 = td_error(1.0, 5.0, 4.0, 1.0, gamma=0.9)   # episode ended: no bootstrap
print(f"case 1: {d1:+.2f} (expected +2.80) | case 2: {d2:+.2f} (expected -1.90) | quiz: {d3:+.2f} (expected -0.40) | terminal: {d4:+.2f} (expected -4.00)")
assert abs(d1 - 2.8) < 1e-9 and abs(d2 + 1.9) < 1e-9 and abs(d3 + 0.4) < 1e-9 and abs(d4 + 4.0) < 1e-9
print("PASSED ✔ Part 3")

**Question 2.** Why do we multiply $V(s_{t+1})$ by $(1-d_t)$? What would go wrong with `MultiEnv` if we forgot it?

*Your answer:* …

---
## Part 4 · n-step returns and advantages  *(20 pts)*

> **Slide 14.** A2C uses $n$ real rewards, then trusts the critic for the rest:
> $$G_t = r_t + \gamma r_{t+1} + \dots + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n}), \qquad \hat A_t = G_t - V(s_t)$$
> Computed backward over the rollout (slide 18, lines 4–7). Each step uses as many real rewards as remain in the rollout, so the **first step gets n rewards and the last step gets one**.

**Shapes.** `rewards`, `values`, `dones`: `(n_steps, n_envs)`; `next_value`: `(n_envs,)` = $V$ of the observation reached after the last step.

### TODO 4.1 — `compute_nstep_returns`
1. `G = next_value` (the bootstrap, line 4).
2. For `t` from `n_steps-1` down to `0`: `G = rewards[t] + gamma * G * (1 - dones[t])`; store `returns[t] = G`.
3. `advantages = returns - values`.

In [ ]:
def compute_nstep_returns(rewards, values, dones, next_value, gamma=0.99):
    returns = torch.zeros_like(rewards)
    # ===== TODO 4.1 =====
    raise NotImplementedError("TODO 4.1: n-step returns, backward pass")
    # ===== END TODO =====
    advantages = returns - values
    return returns, advantages

In [ ]:
# ✅ Check Part 4
rew = torch.tensor([[1.0], [0.0], [2.0]]); val = torch.tensor([[0.5], [0.8], [1.0]])
nv = torch.tensor([1.5])

# (a) no episode end inside the rollout: bootstrap from V(s3) = 1.5
ret, adv = compute_nstep_returns(rew, val, torch.zeros(3, 1), nv, gamma=0.9)
print("returns   :", ret.squeeze().numpy().round(4), "expected [3.7135 3.015  3.35  ]")
assert torch.allclose(ret.squeeze(), torch.tensor([3.7135, 3.015, 3.35]), atol=1e-4)
assert torch.allclose(adv.squeeze(), torch.tensor([3.2135, 2.215, 2.35]), atol=1e-4)

# (b) the episode ends at t = 1: no reward or value may leak across the boundary
ret_b, _ = compute_nstep_returns(rew, val, torch.tensor([[0.0], [1.0], [0.0]]), nv, gamma=0.9)
print("with done :", ret_b.squeeze().numpy().round(4), "expected [1.   0.   3.35]")
assert torch.allclose(ret_b.squeeze(), torch.tensor([1.0, 0.0, 3.35]), atol=1e-4)

# (c) the n-step advantage equals a discounted sum of TD errors (links Part 3 to Part 4)
nxt_vals = torch.cat([val[1:], nv.unsqueeze(0)])
deltas = td_error(rew, val, nxt_vals, torch.zeros(3, 1), gamma=0.9).squeeze()
sum_deltas = torch.tensor([sum(0.9 ** k * deltas[t + k] for k in range(3 - t)) for t in range(3)])
assert torch.allclose(adv.squeeze(), sum_deltas, atol=1e-4)
print("n-step advantage == Σ γᵏ δₜ₊ₖ ✔")
print("PASSED ✔ Part 4")

**Question 3.** In check (b), why is $G_0 = 1.0$ and $G_1 = 0.0$? What does check (c) tell you about the relation between TD errors and the n-step advantage?

*Your answer:* …

---
## Part 5 · The A2C loss  *(15 pts)*

> **Slide 17.**
> $$L = \underbrace{-\hat{\mathbb E}\big[\log \pi_\theta(a_t|s_t)\,\hat A_t\big]}_{\text{policy}} + c_v\,\underbrace{\hat{\mathbb E}\big[(G_t - V_\psi(s_t))^2\big]}_{\text{value}} - c_e\,\underbrace{\hat{\mathbb E}\big[H(\pi_\theta(\cdot|s_t))\big]}_{\text{entropy}}$$

⚠️ **Important:** in the policy term the advantage must be treated as a **constant** — use `advantages.detach()`. Otherwise the policy loss would also push the critic, in the wrong direction.

### TODO 5.1 — `a2c_loss`
Return the total loss and a dictionary with the three components as Python floats.

In [ ]:
def a2c_loss(logprob, advantages, value, returns, entropy, vf_coef=0.5, ent_coef=0.01):
    # ===== TODO 5.1 =====
    # policy_loss = ...   (remember .detach() on the advantages)
    # value_loss  = ...
    # ent         = ...
    # loss        = ...
    raise NotImplementedError("TODO 5.1: implement the A2C loss")
    # ===== END TODO =====
    return loss, {"policy_loss": policy_loss.item(), "value_loss": value_loss.item(), "entropy": ent.item()}

In [ ]:
# ✅ Check 5.1
# (a) numerical value
lp = torch.log(torch.tensor([0.5, 0.25])); A = torch.tensor([2.0, -1.0])
V = torch.tensor([1.0, 3.0]); G = torch.tensor([3.0, 2.0]); H = torch.tensor([0.6, 0.4])
loss, info = a2c_loss(lp, A, V, G, H, vf_coef=0.5, ent_coef=0.01)
exp_policy = -(np.log(0.5) * 2.0 + np.log(0.25) * -1.0) / 2
expected = exp_policy + 0.5 * ((2.0 ** 2 + 1.0 ** 2) / 2) - 0.01 * 0.5
print(f"loss = {loss.item():.4f} (expected {expected:.4f})", info)
assert abs(loss.item() - expected) < 1e-5

# (b) the policy term must not send gradients into the critic through Â = G − V
V2 = torch.tensor([1.0, 3.0], requires_grad=True)
lp2 = lp.clone().requires_grad_(True)           # stands in for the actor's output
adv2 = G - V2                                   # depends on the critic!
loss2, _ = a2c_loss(lp2, adv2, V2.detach(), G, H, vf_coef=0.0, ent_coef=0.0)   # policy term only
loss2.backward()
assert V2.grad is None or torch.all(V2.grad == 0), "gradient leaked into the critic: did you detach the advantages?"
print("advantages correctly detached ✔")
print("PASSED ✔ Check 5.1")

**Question 4.** What would happen if you forgot `.detach()` on the advantages? Why is the entropy term *subtracted*?

*Your answer:* …

---
## Part 6 · Collecting a rollout  *(10 pts)*

> **Slides 16 and 19 — Collect.** All `n_envs` workers take `n_steps` steps in lockstep with the current policy.

### TODO 6.1 — fill in the loop body
At each step `t`: store `obs`; **without gradients** sample `action` (and get `value`) from the agent; step the environments; store `action`, `value`, `reward`, `done`; move to the next observation.

In [ ]:
def collect_rollout(envs: MultiEnv, agent: Agent, obs: torch.Tensor, n_steps: int):
    n = envs.n
    buf = {
        "obs":     torch.zeros((n_steps, n, envs.obs_dim)),
        "actions": torch.zeros((n_steps, n), dtype=torch.long),
        "values":  torch.zeros((n_steps, n)),
        "rewards": torch.zeros((n_steps, n)),
        "dones":   torch.zeros((n_steps, n)),
    }
    for t in range(n_steps):
        # ===== TODO 6.1 =====
        raise NotImplementedError("TODO 6.1: fill the rollout buffer")
        # ===== END TODO =====
    with torch.no_grad():
        next_value = agent.get_value(obs)
    return buf, obs, next_value

In [ ]:
# ✅ Check Part 6
set_seed(0)
_envs = MultiEnv("CartPole-v1", n=4, seed=0)
_agent = Agent(_envs.obs_dim, _envs.n_actions)
_obs = torch.from_numpy(_envs.reset())
_buf, _next_obs, _nv = collect_rollout(_envs, _agent, _obs, n_steps=40)
assert _buf["obs"].shape == (40, 4, 4) and _nv.shape == (4,)
assert (_buf["rewards"] == 1.0).all(), "CartPole gives +1 every step"
assert _buf["values"].abs().sum() > 0, "values were not stored"
assert _buf["dones"].sum() >= 1, "with a near-random policy some episode should end within 40 steps"
assert not torch.equal(_buf["obs"][0], _buf["obs"][1]), "observations must change from step to step"
_envs.close()
print("PASSED ✔ Part 6")

---
## Part 7 · The A2C update and training  *(10 pts)*

> **Slide 18, lines 8–9.** Evaluate the stored observations **with gradients**, build the loss, take **one** optimizer step, then discard the batch.

### TODO 7.1 — `a2c_update`
1. Flatten the buffer: `(n_steps, n_envs, …) → (n_steps * n_envs, …)` (provided).
2. `_, logprob, entropy, value = agent.get_action_and_value(b_obs, b_actions)`.
3. `advantages = b_returns - value` (the loss detaches them for the policy term).
4. Compute `a2c_loss(...)`, then `optimizer.zero_grad()`, `loss.backward()`, clip the gradient norm to `max_grad_norm`, `optimizer.step()`.

In [ ]:
def a2c_update(agent, optimizer, buf, returns, cfg):
    b_obs = buf["obs"].reshape(-1, buf["obs"].shape[-1])
    b_actions = buf["actions"].reshape(-1)
    b_returns = returns.reshape(-1)
    # ===== TODO 7.1 =====
    raise NotImplementedError("TODO 7.1: one A2C gradient step")
    # ===== END TODO =====
    return info

In [ ]:
# ✅ Check 7.1 — one update must change the parameters and lower the loss on the same batch
set_seed(0)
_envs = MultiEnv("CartPole-v1", n=4, seed=0)
_agent = Agent(_envs.obs_dim, _envs.n_actions)
_opt = optim.Adam(_agent.parameters(), lr=1e-2)
_obs = torch.from_numpy(_envs.reset())
_buf, _, _nv = collect_rollout(_envs, _agent, _obs, n_steps=32)
_ret, _ = compute_nstep_returns(_buf["rewards"], _buf["values"], _buf["dones"], _nv)
_cfg = dict(vf_coef=0.5, ent_coef=0.0, max_grad_norm=0.5)
_before = [p.detach().clone() for p in _agent.parameters()]
_info1 = a2c_update(_agent, _opt, _buf, _ret, _cfg)
assert any(not torch.equal(b, p) for b, p in zip(_before, _agent.parameters())), "parameters did not change"
_info2 = a2c_update(_agent, _opt, _buf, _ret, _cfg)
print(f"value loss: {_info1['value_loss']:.3f} → {_info2['value_loss']:.3f}")
assert _info2["value_loss"] < _info1["value_loss"], "the critic should improve on the same batch"
_envs.close()
print("PASSED ✔ Check 7.1")

### Provided: the training loop
Match it to the cycle of slide 19: **Collect → Evaluate → Update once → Discard**. The optimizer is RMSprop, as in the A3C paper and Stable-Baselines3.

Default settings follow Stable-Baselines3 (`n_steps=5`, `lr=7e-4`, `gamma=0.99`, `vf_coef=0.5`, `max_grad_norm=0.5`), with 8 parallel environments. We use `ent_coef=0.01` (as in the A3C paper; SB3's default is 0.0), which makes CartPole training a little more robust.

In [ ]:
DEFAULT_CFG = dict(
    env_id="CartPole-v1", seed=1, total_steps=250_000,
    n_envs=8, n_steps=5,
    lr=7e-4, gamma=0.99, vf_coef=0.5, ent_coef=0.01, max_grad_norm=0.5,
)

def train(**overrides):
    cfg = {**DEFAULT_CFG, **overrides}
    set_seed(cfg["seed"])
    envs = MultiEnv(cfg["env_id"], cfg["n_envs"], seed=cfg["seed"])
    agent = Agent(envs.obs_dim, envs.n_actions)
    optimizer = optim.RMSprop(agent.parameters(), lr=cfg["lr"], alpha=0.99, eps=1e-5)

    obs = torch.from_numpy(envs.reset())
    batch = cfg["n_envs"] * cfg["n_steps"]
    n_iters = cfg["total_steps"] // batch
    log_every = max(1, n_iters // 20)
    history = {"step": [], "mean_return": [], "entropy": [], "value_loss": []}
    t0 = time.time()
    for it in range(1, n_iters + 1):
        buf, obs, next_value = collect_rollout(envs, agent, obs, cfg["n_steps"])              # 1 · Collect
        returns, _ = compute_nstep_returns(buf["rewards"], buf["values"], buf["dones"],        # 2 · Evaluate
                                           next_value, cfg["gamma"])
        info = a2c_update(agent, optimizer, buf, returns, cfg)                                 # 3 · Update once
        # 4 · Discard: the next iteration overwrites buf
        if it % log_every == 0 or it == n_iters:
            recent = envs.finished_returns[-20:]
            history["step"].append(it * batch)
            history["mean_return"].append(float(np.mean(recent)) if recent else 0.0)
            history["entropy"].append(info["entropy"]); history["value_loss"].append(info["value_loss"])
            print(f"step {it * batch:>7} | mean return (last 20 ep) {history['mean_return'][-1]:6.1f} | "
                  f"entropy {info['entropy']:.3f} | value loss {info['value_loss']:8.3f} | {time.time() - t0:5.1f}s")
    envs.close()
    return agent, history

### 7.2 · Train your agent
This takes a few minutes on a laptop CPU. A correct implementation usually climbs well above the random baseline (~20) and often reaches the maximum of 500; A2C curves are noticeably **noisier** than PPO's — keep that in mind for the next lab.

In [ ]:
agent, hist = train()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].plot(hist["step"], hist["mean_return"], color="#E8833A"); axes[0].axhline(500, ls=":", color="gray")
axes[0].set_title("Mean episode return (last 20)")
axes[1].plot(hist["step"], hist["entropy"], color="#2F6FD6"); axes[1].set_title("Policy entropy")
axes[2].plot(hist["step"], hist["value_loss"], color="#B8561A"); axes[2].set_title("Critic value loss"); axes[2].set_yscale("log")
for ax in axes: ax.set_xlabel("environment steps")
plt.tight_layout(); plt.show()

In [ ]:
def evaluate(agent, env_id="CartPole-v1", episodes=10, seed=1000):
    env = gym.make(env_id); scores = []
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep); done, total = False, 0.0
        while not done:
            with torch.no_grad():
                logits = agent.actor(torch.tensor(obs, dtype=torch.float32))
            obs, r, term, trunc, _ = env.step(int(torch.argmax(logits)))
            total += r; done = term or trunc
        scores.append(total)
    env.close()
    return float(np.mean(scores)), float(np.std(scores))

m, s = evaluate(agent)
print(f"Greedy evaluation over 10 episodes: {m:.1f} ± {s:.1f}")

---
## Part 8 · Experiments  *(10 pts)*

### TODO 8.1 — the rollout length $n$ (slide 14: bias vs variance)
Train three agents with `n_steps` = **1**, **5** and **32** (`total_steps=150_000`, everything else default) and plot the three learning curves on one figure.

In [ ]:
# ===== TODO 8.1 =====
# n_results = {}
# for n in [1, 5, 32]:
#     _, h = train(n_steps=n, total_steps=150_000)
#     n_results[n] = h
# ... plot the curves
# ===== END TODO =====

**Question 5.** Compare the three runs. Explain the differences using the bias–variance trade-off (slide 14). Also consider how many gradient updates each setting performs for the same number of environment steps.

*Your answer:* …

### TODO 8.2 — the fragile step size (slide 21: what A2C does *not* fix)
Train with `lr` = **7e-4** (default) and **7e-3** (10× larger), `total_steps=150_000`. Plot both curves.

In [ ]:
# ===== TODO 8.2 =====
# ===== END TODO =====

**Question 6.** What happens with the larger learning rate? Using slide 21, explain why A2C is fragile to the step size and what kind of mechanism would fix it (hint: next lecture).

*Your answer:* …

---
## Reflection questions
Answer briefly (2–4 sentences each).

**R1.** Why does subtracting $V(s)$ reduce variance without changing what the policy should learn? *(hint: $\sum_a \pi(a|s) = 1$)*

**R2.** Why is the batch discarded after one update?

**R3.** Why does A2C use parallel environments instead of a replay buffer?

**R4.** What is the TD error for r = 1, V(s) = 5, V(s′) = 4, γ = 0.9?

**R5.** Which part(s) of `train()` would you need to change to turn A2C into PPO?

---
### Submission
Submit this notebook **with all outputs** (Kernel → Restart & Run All, then save). Every ✅ check must print `PASSED` and the plots of Parts 7–8 must be visible.

### References
* Hugging Face Deep RL Course — Unit 4 (Policy Gradient) and Unit 6 (Actor–Critic, A2C): https://huggingface.co/learn/deep-rl-course/unit6/introduction
* V. Mnih et al., "Asynchronous Methods for Deep Reinforcement Learning" (A3C), ICML 2016.
* Stable-Baselines3 A2C documentation: https://stable-baselines3.readthedocs.io/en/master/modules/a2c.html
* M. Hu, *The Art of Reinforcement Learning*, Apress, 2023.
* S. Huang et al., "A2C is a special case of PPO", 2022.